In [1]:
import dspy
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

#Define the differnet Models and Providers

localgpt = dspy.LM(
    model = "ollama/gpt-oss",
    api_base = "http://localhost:11434",
    api_key = "ollama"
)

localglm = dspy.LM(
    model = "ollama/glm-4.7-flash:q8_0",
    api_base = "http://localhost:11434",
    api_key = "ollama"
)

cloudgpt = dspy.LM(
    model = "openai/gpt-5.4",
    api_base = "https://api.openai.com/v1",
    api_key = OPENAI_API_KEY
)

cloudclaude = dspy.LM(
    model = "anthropic/claude-opus-4-6",
    #api_base = "https://api.anthropic.com",
    api_key = ANTHROPIC_API_KEY
)

models = [localgpt, localglm, cloudgpt, cloudclaude]

#define the Signatures

dspy.settings.configure(lm = models[0])

class Explain(dspy.Signature):
    topic = dspy.InputField()
    explanation = dspy.OutputField()

class Summarise(dspy.Signature):
    text = dspy.InputField()
    summary = dspy.OutputField()

class ConfidenceScore(dspy.Signature):
    question = dspy.InputField()
    answer = dspy.InputField()
    confidence = dspy.OutputField(desc = "A confidence score from 0 to 100")


class Explain_Summarise(dspy.Module):
    def __init__(self):
        self.explain = dspy.Predict(Explain)
        self.summarise = dspy.Predict(Summarise)

    def forward(self, topic):
        explanation = self.explain(topic = topic).explanation
        summary = self.summarise(text = explanation).summary
        return summary

In [2]:
def safe_float(x, default=0.0):
    try:
        return float(str(x).strip())
    except:
        return default

# Equal weights by default.
# You can change these later if you want some models to count more than others.
model_weights = {
    "ollama/gpt-oss": .05,
    "ollama/glm-4.7-flash:q8_0": .25,
    "openai/gpt-5.4": .75,
    "anthropic/claude-opus-4-6": .80
}

all_outputs = []

In [3]:
#Define the Variables
question = input("What do you want to know? ")
print("-------------------------------------------------------------------------------------------")
for model in models:
    with dspy.context(lm = model):
        pipeline = Explain_Summarise()
        answer = pipeline(topic = question)

        all_outputs.append({
            "generator_model": model,
            "generator_name": model.model,
            "answer": answer
        })

print(all_outputs)
print("-------------------------------------------------------------------------------------------")

What do you want to know?  What is the difference between GEPA and JEPA


-------------------------------------------------------------------------------------------
[{'generator_model': <dspy.clients.lm.LM object at 0x741c5f62eb10>, 'generator_name': 'ollama/gpt-oss', 'answer': 'GEPA (EU‑Canada) and JEPA (EU‑Japan) are two major trade agreements aimed at expanding EU trade ties beyond Europe. Both cover goods, services, investment, and regulatory cooperation, but they differ in focus and implementation. GEPA emphasizes sustainability, environmental standards, and the green and digital transition, with tariff‑free access across a wide product range and ongoing phase‑in of tariff reductions. JEPA, signed in 2019 and in force since 2021, centers on industrial goods, digital trade, e‑commerce, data protection, and IP, and has largely completed its tariff‑elimination schedule. While GEPA is still in the process of implementing many provisions, JEPA is already active with significant tariff cuts. In short, GEPA prioritizes eco‑friendly and digital initiatives wit

In [4]:
for output in all_outputs:
    confidence_scores = []
    weighted_sum = 0.0
    total_weight = 0.0

    for scorer_model in models:
        with dspy.context(lm = scorer_model):
            scorer = dspy.Predict(ConfidenceScore)

            score_result = scorer(
                question = question,
                answer = str(output["answer"])
            )

            score = safe_float(score_result.confidence, default = 0.0)
            weight = model_weights.get(scorer_model.model, 1.0)

            confidence_scores.append({
                "scorer_model": scorer_model.model,
                "score": score,
                "weight": weight
            })

            weighted_sum += score * weight
            total_weight += weight
    
    weighted_confidence = weighted_sum / total_weight if total_weight > 0 else 0.0
    
    print(f"Model: {output['generator_name']}")
    print(f"Summary: {output['answer']}")
    print(f"Confidence Score: {weighted_confidence:.2f}/100")
    print("Individual Scores:")
    for item in confidence_scores:
        print(f"  - {item['scorer_model']}: {item['score']:.2f} (weight={item['weight']})")
    print("-------------------------------------------------------------------------------------------")
            

Model: ollama/gpt-oss
Summary: GEPA (EU‑Canada) and JEPA (EU‑Japan) are two major trade agreements aimed at expanding EU trade ties beyond Europe. Both cover goods, services, investment, and regulatory cooperation, but they differ in focus and implementation. GEPA emphasizes sustainability, environmental standards, and the green and digital transition, with tariff‑free access across a wide product range and ongoing phase‑in of tariff reductions. JEPA, signed in 2019 and in force since 2021, centers on industrial goods, digital trade, e‑commerce, data protection, and IP, and has largely completed its tariff‑elimination schedule. While GEPA is still in the process of implementing many provisions, JEPA is already active with significant tariff cuts. In short, GEPA prioritizes eco‑friendly and digital initiatives with Canada, whereas JEPA focuses on digital trade and technology cooperation with Japan.
Confidence Score: 51.75/100
Individual Scores:
  - ollama/gpt-oss: 95.00 (weight=1.0)
  -